# PSB lab: sensor anomaly detection with SageMaker Random Cut Forest

Run the cells in order. Cells 1-4 (Lab 06 Task 2) load the backfilled readings and train the model. Cells 5-10 (Lab 06 Task 3) deploy a serverless endpoint, score the data, measure the model and print `ANOMALY_CUTOFF`.

## Cell 1: Imports, lab bucket and SageMaker session

Set `BUCKET` to your lab bucket name (psb-lab-<DeploymentID>), then run the cell.

In [ ]:
import json
import concurrent.futures as cf

import boto3
import numpy as np
import pandas as pd
import sagemaker
from sagemaker import get_execution_role

BUCKET = "psb-lab-<DEPLOYMENT-ID>"  # replace with your lab bucket, e.g. psb-lab-123456

session = sagemaker.Session()
role = get_execution_role()
region = session.boto_region_name
s3 = boto3.client("s3", region_name=region)

print("Region :", region)
print("Role   :", role)
print("Bucket :", BUCKET)
print("SageMaker SDK:", sagemaker.__version__)

## Cell 2: Load raw telemetry from S3

Lists every JSON object under `raw/` (written by the IoT rule) and loads up to 20,000 readings into a pandas DataFrame.

In [ ]:
MAX_RECORDS = 20000

keys = []
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=BUCKET, Prefix="raw/"):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".json"):
            keys.append(obj["Key"])
keys = sorted(keys)[-MAX_RECORDS:]
print(f"Found {len(keys)} raw objects (using up to {MAX_RECORDS})")

def load(key):
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    try:
        return json.loads(body)
    except ValueError:
        return None

with cf.ThreadPoolExecutor(max_workers=32) as pool:
    records = [r for r in pool.map(load, keys) if isinstance(r, dict)]

df = pd.DataFrame(records)
df = df.sort_values("timestamp").reset_index(drop=True)
print(f"Loaded {len(df)} readings")
df.head()

## Cell 3: Build features and hold labels aside

The model only sees the 4 sensor values. The `injected_anomaly` flags are kept aside to measure the model later.

In [ ]:
FEATURES = ["temperature_c", "humidity_pct", "co2_ppm", "vibration_g"]

df = df.dropna(subset=FEATURES).reset_index(drop=True)
features = df[FEATURES].astype("float32")
labels = df.get("injected_anomaly", pd.Series(False, index=df.index)).fillna(False).astype(bool)

print(f"Rows: {len(features)} | injected anomalies: {labels.sum()} ({100 * labels.mean():.2f}%)")
features.describe()

## Cell 4: Train the Random Cut Forest model

Trains the built-in SageMaker Random Cut Forest algorithm on one ml.m5.large instance. This takes about 5 to 8 minutes.

In [ ]:
from sagemaker import RandomCutForest

rcf = RandomCutForest(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    num_trees=64,
    num_samples_per_tree=256,
    data_location=f"s3://{BUCKET}/model/rcf-input/",
    output_path=f"s3://{BUCKET}/model/",
    sagemaker_session=session,
)

rcf.fit(rcf.record_set(features.values.astype("float32")))
print("Model artifact:", rcf.model_data)

## Cell 5: Deploy a serverless endpoint

Deploys the model to the serverless endpoint `psb-anomaly-endpoint` (2048 MB, max concurrency 5). Takes about 3 to 5 minutes. If the endpoint already exists, the cell attaches to it.

In [ ]:
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer
from sagemaker.serverless import ServerlessInferenceConfig

ENDPOINT_NAME = "psb-anomaly-endpoint"
serverless_config = ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=5)

try:
    predictor = rcf.deploy(
        serverless_inference_config=serverless_config,
        endpoint_name=ENDPOINT_NAME,
        serializer=CSVSerializer(),
        deserializer=JSONDeserializer(),
    )
except Exception as exc:
    if "already exist" not in str(exc):
        raise
    print(f"Endpoint {ENDPOINT_NAME} already exists, attaching to it.")
    predictor = Predictor(
        endpoint_name=ENDPOINT_NAME,
        sagemaker_session=session,
        serializer=CSVSerializer(),
        deserializer=JSONDeserializer(),
    )

print("Endpoint ready:", predictor.endpoint_name)

## Cell 6: Score every reading

Sends the readings to the endpoint in batches of 500. Higher scores mean more unusual readings.

In [ ]:
BATCH = 500
scores = []
values = features.values
for start in range(0, len(values), BATCH):
    result = predictor.predict(values[start:start + BATCH])
    scores.extend(r["score"] for r in result["scores"])
    print(f"scored {min(start + BATCH, len(values))}/{len(values)}", end="\r")

scores = np.array(scores, dtype=float)
df["anomaly_score"] = scores
print(f"\nScores: min={scores.min():.4f} mean={scores.mean():.4f} max={scores.max():.4f}")

## Cell 7: Pick the anomaly cutoff

Readings above the 98th percentile of scores are flagged as anomalies.

In [ ]:
cutoff = float(np.percentile(scores, 98))
flagged = scores > cutoff
df["flagged"] = flagged

print(f"Cutoff (98th percentile): {cutoff:.4f}")
print(f"Flagged: {flagged.sum()} of {len(flagged)} readings")

## Cell 8: Measure precision, recall and F1

Compares the flagged readings with the anomalies the simulator injected.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

y_true = labels.values
y_pred = flagged

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[False, True]).ravel()

print(f"Precision: {precision:.3f}  (of the flagged readings, how many were real anomalies)")
print(f"Recall   : {recall:.3f}  (of the real anomalies, how many were flagged)")
print(f"F1 score : {f1:.3f}")
print(f"TP={tp} FP={fp} FN={fn} TN={tn}")

## Cell 9: Plot the scores

Blue line: anomaly score per reading. Dashed red line: the cutoff. Orange dots: injected anomalies.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(np.arange(len(scores)), scores, lw=0.6, color="#2563eb", label="anomaly score")
ax.axhline(cutoff, color="red", ls="--", lw=1.2, label=f"cutoff {cutoff:.3f}")
idx = np.where(y_true)[0]
ax.scatter(idx, scores[idx], s=14, color="orange", zorder=3, label="injected anomaly")
ax.set_xlabel("reading (time order)")
ax.set_ylabel("RCF score")
ax.set_title("PSB lab: Random Cut Forest anomaly scores")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Cell 10: Save scored data and print the cutoff

Writes `curated/scored/backfill-scored.csv` to the lab bucket and prints `ANOMALY_CUTOFF`. Copy that value: you need it for the `psb-score-telemetry` Lambda in Lab 07.

In [ ]:
out_key = "curated/scored/backfill-scored.csv"
csv_body = df[["timestamp", "device", "location", *FEATURES, "injected_anomaly", "anomaly_score", "flagged"]].to_csv(index=False)
s3.put_object(Bucket=BUCKET, Key=out_key, Body=csv_body.encode("utf-8"), ContentType="text/csv")
print(f"Wrote s3://{BUCKET}/{out_key} ({len(df)} rows)")
print()
print(f"ANOMALY_CUTOFF = {cutoff:.4f}")
print("Copy the ANOMALY_CUTOFF value above. You will paste it into the psb-score-telemetry Lambda environment variables in Lab 07.")